# Train the CelebA diffusion model on Google Colab

This notebook clones [`5cisummai/diffusion`](https://github.com/5cisummai/diffusion), extracts CelebA from `archive.zip` onto Colab's fast local disk, trains `train_celeb.py`, and saves checkpoints, logs, and samples to Google Drive.

Before running: select **Runtime → Change runtime type → T4 GPU** (or another CUDA GPU). The default run is 20 epochs and may take a long time.

## 1. Mount Google Drive and configure paths

```text
MyDrive/diffusion/
├── archive.zip        # CelebA zip (source only)
└── outputs_celeb/     # train.log, checkpoint.pt, and samples

/content/data/celeba/  # extracted dataset (local Colab disk, faster for training)
```

CelebA stays on local disk for speed. Checkpoints and samples are saved to Drive. If the runtime disconnects, re-run section 4 to re-extract.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

REPO_URL = 'https://github.com/5cisummai/diffusion.git'
REPO_DIR = Path('/content/diffusion')
DRIVE_ROOT = Path('/content/drive/MyDrive/diffusion')
LOCAL_DATA_DIR = Path('/content/data')
OUTPUT_DIR = DRIVE_ROOT / 'outputs_celeb'

LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Repository: {REPO_DIR}')
print(f'CelebA data (local): {LOCAL_DATA_DIR}')
print(f'Training outputs (Drive): {OUTPUT_DIR}')

## 2. Clone or refresh the repository

The refresh branch makes the notebook safe to rerun in the same Colab session.

In [ ]:
import subprocess

if (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    if REPO_DIR.exists():
        raise RuntimeError(f'{REPO_DIR} exists but is not a Git checkout. Remove it and rerun this cell.')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

print('Repository is ready.')
print((REPO_DIR / 'train_celeb.py').resolve())

## 3. Install dependencies and check the GPU

In [4]:
%pip install -q -r /content/diffusion/requirements.txt

import torch

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU not detected. In Colab, choose Runtime → Change runtime type → T4 GPU, then rerun the notebook.')

print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'CUDA: {torch.version.cuda}')

PyTorch: 2.11.0+cu128
GPU: NVIDIA A100-SXM4-40GB
CUDA: 12.8


## 4. Extract CelebA from `archive.zip`

Upload your Kaggle zip to **`MyDrive/diffusion/archive.zip`**, then run the cell below.

It copies the zip to `/content/`, unzips locally, and keeps the dataset at **`/content/data/celeba/`** (not Drive). Reruns in the same session skip extraction when at least 200k images are already present.

In [5]:
import csv
import shutil
import zipfile
from pathlib import Path

CELEBA_DIR = LOCAL_DATA_DIR / 'celeba'
CELEBA_ZIP_DRIVE = DRIVE_ROOT / 'archive.zip'
CELEBA_ZIP_LOCAL = Path('/content/archive.zip')
STAGING_DIR = Path('/content/celeba_unzip')
MIN_CELEBA_IMAGES = 200_000
LIST_FILE_STEMS = [
    'list_attr_celeba',
    'list_eval_partition',
    'list_bbox_celeba',
    'list_landmarks_align_celeba',
]
LIST_FILES = [f'{stem}.txt' for stem in LIST_FILE_STEMS]
REQUIRED_MARKERS = [CELEBA_DIR / 'img_align_celeba', *[CELEBA_DIR / name for name in LIST_FILES]]


def count_celeba_images():
    image_dir = CELEBA_DIR / 'img_align_celeba'
    if not image_dir.is_dir():
        return 0
    return sum(1 for _ in image_dir.glob('*.jpg'))


def celeba_is_ready():
    if not all(path.exists() for path in REQUIRED_MARKERS):
        return False
    return count_celeba_images() >= MIN_CELEBA_IMAGES


def normalize_filename(name: str) -> str:
    name = name.strip()
    if name.endswith('.jpg'):
        return name
    if name.isdigit():
        return f'{int(name):06d}.jpg'
    return name


def find_metadata_file(staging_dir: Path, stem: str) -> Path:
    for ext in ('.txt', '.csv'):
        matches = list(staging_dir.glob(f'**/{stem}{ext}'))
        if matches:
            return matches[0]
    raise RuntimeError(f'Missing {stem}.txt or {stem}.csv in extracted zip.')


def convert_csv_to_celeba_txt(csv_path: Path, txt_path: Path, stem: str):
    with csv_path.open(newline='', encoding='utf-8') as handle:
        reader = csv.reader(handle)
        header = next(reader)
        rows = [row for row in reader if row]

    if stem == 'list_attr_celeba':
        attr_names = header[1:]
        lines = [str(len(rows)), ' '.join(attr_names)]
        for row in rows:
            values = [normalize_filename(row[0]), *row[1:]]
            lines.append(' '.join(values))
    elif stem == 'list_eval_partition':
        lines = [str(len(rows))]
        for row in rows:
            lines.append(f'{normalize_filename(row[0])} {row[1].strip()}')
    elif stem == 'list_bbox_celeba':
        lines = [str(len(rows))]
        for row in rows:
            lines.append(' '.join([normalize_filename(row[0]), *row[1:10]]))
    elif stem == 'list_landmarks_align_celeba':
        lines = [str(len(rows))]
        for row in rows:
            lines.append(' '.join([normalize_filename(row[0]), *row[1:11]]))
    else:
        raise ValueError(f'Unsupported metadata file: {stem}')

    txt_path.write_text('\n'.join(lines) + '\n')


def install_metadata_file(staging_dir: Path, dest_dir: Path, stem: str):
    source = find_metadata_file(staging_dir, stem)
    dest = dest_dir / f'{stem}.txt'
    if source.suffix == '.csv':
        print(f'Converting {source.name} -> {dest.name}...')
        convert_csv_to_celeba_txt(source, dest, stem)
    else:
        shutil.copy2(source, dest)


def find_best_image_dir(staging_dir: Path) -> Path | None:
    best_dir = None
    best_count = 0
    for candidate in staging_dir.rglob('*'):
        if not candidate.is_dir():
            continue
        if candidate.name not in {'img_align_celeba', 'image_align_celeba'}:
            continue
        jpg_count = sum(1 for _ in candidate.glob('*.jpg'))
        if jpg_count > best_count:
            best_count = jpg_count
            best_dir = candidate
    return best_dir


def install_celeba_from_staging(staging_dir: Path, dest_dir: Path):
    image_dir = find_best_image_dir(staging_dir)
    if image_dir is None:
        raise RuntimeError(
            'Could not find CelebA images in extracted zip. '
            'Expected a folder named img_align_celeba or image_align_celeba.'
        )

    if dest_dir.exists():
        shutil.rmtree(dest_dir)
    dest_dir.mkdir(parents=True)

    print(f'Installing images from {image_dir} -> {dest_dir / "img_align_celeba"}...')
    shutil.copytree(image_dir, dest_dir / 'img_align_celeba')

    for stem in LIST_FILE_STEMS:
        install_metadata_file(staging_dir, dest_dir, stem)


image_count = count_celeba_images()
if celeba_is_ready():
    print(f'CelebA already present at {CELEBA_DIR} ({image_count:,} images). Skipping extraction.')
else:
    if not CELEBA_ZIP_DRIVE.exists():
        raise FileNotFoundError(
            f'Missing {CELEBA_ZIP_DRIVE}. Upload archive.zip to MyDrive/diffusion/ and rerun.'
        )

    if CELEBA_DIR.exists():
        print(
            f'Found incomplete CelebA folder at {CELEBA_DIR} '
            f'({image_count:,} images). Removing it and extracting from zip...'
        )
        shutil.rmtree(CELEBA_DIR)

    if STAGING_DIR.exists():
        shutil.rmtree(STAGING_DIR)
    STAGING_DIR.mkdir(parents=True)

    if CELEBA_ZIP_LOCAL.exists():
        CELEBA_ZIP_LOCAL.unlink()

    print(f'Copying {CELEBA_ZIP_DRIVE} to {CELEBA_ZIP_LOCAL}...')
    shutil.copy2(CELEBA_ZIP_DRIVE, CELEBA_ZIP_LOCAL)

    print(f'Extracting {CELEBA_ZIP_LOCAL} on local disk...')
    with zipfile.ZipFile(CELEBA_ZIP_LOCAL) as archive:
        archive.extractall(STAGING_DIR)

    install_celeba_from_staging(STAGING_DIR, CELEBA_DIR)

    shutil.rmtree(STAGING_DIR)
    if CELEBA_ZIP_LOCAL.exists():
        CELEBA_ZIP_LOCAL.unlink()

    if not celeba_is_ready():
        image_count = count_celeba_images()
        raise RuntimeError(
            f'Extraction finished but dataset is incomplete in {CELEBA_DIR} '
            f'({image_count:,} images found, expected at least {MIN_CELEBA_IMAGES:,}).'
        )

    image_count = count_celeba_images()
    print(f'CelebA ready at {CELEBA_DIR} ({image_count:,} images).')

Found incomplete CelebA folder at /content/data/celeba (202,599 images). Removing it and extracting from zip...
Copying /content/drive/MyDrive/diffusion/archive.zip to /content/archive.zip...
Extracting /content/archive.zip on local disk...
Installing images from /content/celeba_unzip/img_align_celeba/img_align_celeba -> /content/data/celeba/img_align_celeba...
Converting list_attr_celeba.csv -> list_attr_celeba.txt...
Converting list_eval_partition.csv -> list_eval_partition.txt...
Converting list_bbox_celeba.csv -> list_bbox_celeba.txt...
Converting list_landmarks_align_celeba.csv -> list_landmarks_align_celeba.txt...
CelebA ready at /content/data/celeba (202,599 images).


## 5. Set training parameters

Set `RESUME = True` to continue from `outputs_celeb/checkpoint.pt` when it exists. Set it to `False` to start a fresh run; the existing output directory will still be used, so copy or rename old results first if you want to preserve them.

In [ ]:
EPOCHS = 20
BATCH_SIZE = 64
LEARNING_RATE = 2e-4
NUM_WORKERS = 4
NUM_SAMPLES = 16
SAMPLE_STEPS = 50
NUM_RES_BLOCKS = 2
NUM_MID_BLOCKS = 2
RESUME = True
NO_AMP = False

CHECKPOINT_PATH = OUTPUT_DIR / 'checkpoint.pt'
print(f'Epochs: {EPOCHS}')
print(f'Batch size: {BATCH_SIZE}')
print('Resume checkpoint:', CHECKPOINT_PATH if RESUME and CHECKPOINT_PATH.exists() else 'none')

Epochs: 20
Batch size: 64
Resume checkpoint: none


## 6. Train CelebA

Training reads CelebA from local disk (`/content/data/celeba/`). Checkpoints, samples, and logs are saved to Drive.

import os
import subprocess
import sys

TRAIN_CELEB_SOURCE = 'import argparse\nimport csv\nfrom pathlib import Path\n\nimport torch\nimport torch.nn.functional as F\nfrom PIL import Image\nfrom torch.utils.data import DataLoader, Dataset\nfrom torchvision import transforms\nfrom torchvision.utils import save_image\nfrom tqdm import tqdm\n\nfrom model_large import UnetLarge\nfrom noise_scheduler import LinearNoiseScheduler\n\nNUM_TIMESTEPS = 1000\nBETA_START = 0.0001\nBETA_END = 0.02\nIMAGE_SIZE = 128\nIN_CHANNELS = 3\n\n\ndef get_device():\n    if torch.cuda.is_available():\n        return torch.device("cuda")\n    if torch.backends.mps.is_available():\n        return torch.device("mps")\n    return torch.device("cpu")\n\n\ndef use_bf16(device: torch.device) -> bool:\n    return device.type == "cuda" and torch.cuda.is_bf16_supported()\n\n\nMIN_CELEBA_IMAGES = 200_000\n\n\ndef count_celeba_images(celeba_dir: Path) -> int:\n    image_dir = celeba_dir / "img_align_celeba"\n    if not image_dir.is_dir():\n        return 0\n    return sum(1 for _ in image_dir.glob("*.jpg"))\n\n\ndef normalize_filename(name: str) -> str:\n    name = name.strip()\n    if name.endswith(".jpg"):\n        return name\n    if name.isdigit():\n        return f"{int(name):06d}.jpg"\n    return name\n\n\ndef partition_file_exists(celeba_dir: Path) -> bool:\n    return (celeba_dir / "list_eval_partition.txt").exists() or (\n        celeba_dir / "list_eval_partition.csv"\n    ).exists()\n\n\ndef celeba_is_ready(data_dir: str) -> bool:\n    celeba_dir = Path(data_dir) / "celeba"\n    if not (celeba_dir / "img_align_celeba").is_dir():\n        return False\n    if not partition_file_exists(celeba_dir):\n        return False\n    return count_celeba_images(celeba_dir) >= MIN_CELEBA_IMAGES\n\n\ndef load_partition(celeba_dir: Path) -> dict[str, int]:\n    txt_path = celeba_dir / "list_eval_partition.txt"\n    csv_path = celeba_dir / "list_eval_partition.csv"\n    partition: dict[str, int] = {}\n\n    if txt_path.exists():\n        with txt_path.open(encoding="utf-8") as handle:\n            handle.readline()\n            for line in handle:\n                line = line.strip()\n                if not line:\n                    continue\n                filename, split_value = line.rsplit(" ", 1)\n                partition[normalize_filename(filename)] = int(split_value)\n    elif csv_path.exists():\n        with csv_path.open(newline="", encoding="utf-8") as handle:\n            reader = csv.reader(handle)\n            next(reader)\n            for row in reader:\n                if row:\n                    partition[normalize_filename(row[0])] = int(row[1])\n    else:\n        raise FileNotFoundError(f"Missing partition file in {celeba_dir}")\n\n    return partition\n\n\nclass CelebAImageDataset(Dataset):\n    def __init__(self, data_dir: str, split: str, transform=None):\n        celeba_dir = Path(data_dir) / "celeba"\n        self.image_dir = celeba_dir / "img_align_celeba"\n        self.transform = transform\n        split_ids = {"train": 0, "valid": 1, "test": 2}\n        if split not in split_ids:\n            raise ValueError(f"Unknown split: {split}")\n\n        partition = load_partition(celeba_dir)\n        target_split = split_ids[split]\n        self.filenames = sorted(\n            path.name\n            for path in self.image_dir.glob("*.jpg")\n            if partition.get(path.name) == target_split\n        )\n        if not self.filenames:\n            raise RuntimeError(f"No CelebA images found for split={split} under {self.image_dir}")\n\n    def __len__(self) -> int:\n        return len(self.filenames)\n\n    def __getitem__(self, index: int):\n        image_path = self.image_dir / self.filenames[index]\n        image = Image.open(image_path).convert("RGB")\n        if self.transform is not None:\n            image = self.transform(image)\n        return image, 0\n\n\ndef get_celeba_dataloaders(data_dir: str, batch_size: int, num_workers: int):\n    if not celeba_is_ready(data_dir):\n        raise RuntimeError(\n            f"CelebA not found under {Path(data_dir) / \'celeba\'}. "\n            "Extract the dataset first (see train_celeb_colab.ipynb section 4)."\n        )\n\n    transform = transforms.Compose([\n        transforms.Resize(IMAGE_SIZE),\n        transforms.CenterCrop(IMAGE_SIZE),\n        transforms.RandomHorizontalFlip(),\n        transforms.ToTensor(),\n        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),\n    ])\n    eval_transform = transforms.Compose([\n        transforms.Resize(IMAGE_SIZE),\n        transforms.CenterCrop(IMAGE_SIZE),\n        transforms.ToTensor(),\n        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),\n    ])\n\n    train_dataset = CelebAImageDataset(data_dir, split="train", transform=transform)\n    test_dataset = CelebAImageDataset(data_dir, split="valid", transform=eval_transform)\n\n    loader_kwargs = {\n        "batch_size": batch_size,\n        "num_workers": num_workers,\n        "pin_memory": torch.cuda.is_available(),\n    }\n    train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)\n    test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)\n    return train_loader, test_loader\n\n\ndef train_epoch(model, train_loader, scheduler, optimizer, device, epoch, total_epochs, amp_enabled):\n    model.train()\n    total_loss = 0.0\n    progress = tqdm(train_loader, desc=f"Train {epoch}/{total_epochs}", leave=False)\n\n    for images, _ in progress:\n        images = images.to(device, non_blocking=True)\n        batch_size = images.shape[0]\n\n        noise = torch.randn_like(images)\n        timesteps = torch.randint(0, scheduler.num_timesteps, (batch_size,), device=device)\n        noisy_images = scheduler.add_noise(images, noise, timesteps)\n\n        optimizer.zero_grad(set_to_none=True)\n        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=amp_enabled):\n            noise_pred = model(noisy_images, timesteps)\n            loss = F.mse_loss(noise_pred, noise)\n\n        loss.backward()\n        optimizer.step()\n\n        total_loss += loss.item()\n        progress.set_postfix(loss=f"{loss.item():.4f}")\n\n    return total_loss / len(train_loader)\n\n\n@torch.no_grad()\ndef evaluate(model, test_loader, scheduler, device, epoch, total_epochs, amp_enabled):\n    model.eval()\n    total_loss = 0.0\n    progress = tqdm(test_loader, desc=f"Eval {epoch}/{total_epochs}", leave=False)\n\n    for images, _ in progress:\n        images = images.to(device, non_blocking=True)\n        batch_size = images.shape[0]\n\n        noise = torch.randn_like(images)\n        timesteps = torch.randint(0, scheduler.num_timesteps, (batch_size,), device=device)\n        noisy_images = scheduler.add_noise(images, noise, timesteps)\n\n        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=amp_enabled):\n            noise_pred = model(noisy_images, timesteps)\n            loss = F.mse_loss(noise_pred, noise)\n\n        total_loss += loss.item()\n        progress.set_postfix(loss=f"{loss.item():.4f}")\n\n    return total_loss / len(test_loader)\n\n\n@torch.no_grad()\ndef sample_images(\n    model,\n    scheduler,\n    device,\n    num_images: int,\n    output_path: Path,\n    sample_steps: int,\n    amp_enabled: bool,\n):\n    model.eval()\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n\n    images = torch.randn(num_images, IN_CHANNELS, IMAGE_SIZE, IMAGE_SIZE, device=device)\n    ddim_timesteps = scheduler.get_ddim_timesteps(sample_steps, device=device)\n\n    for i in tqdm(range(len(ddim_timesteps) - 1), desc="DDIM sampling", leave=False):\n        t = ddim_timesteps[i].item()\n        t_prev = ddim_timesteps[i + 1].item()\n        t_batch = torch.full((num_images,), t, device=device, dtype=torch.long)\n        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=amp_enabled):\n            noise_pred = model(images, t_batch)\n        images, _ = scheduler.ddim_step(images, noise_pred.float(), t, t_prev, eta=0.0)\n\n    images = (images.clamp(-1, 1) + 1) / 2\n    save_image(images, output_path, nrow=int(num_images ** 0.5))\n\n\ndef save_checkpoint(model, optimizer, scaler, epoch, output_dir: Path):\n    output_dir.mkdir(parents=True, exist_ok=True)\n    torch.save(\n        {\n            "epoch": epoch,\n            "model_state_dict": model.state_dict(),\n            "optimizer_state_dict": optimizer.state_dict(),\n            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,\n        },\n        output_dir / "checkpoint.pt",\n    )\n\n\ndef load_checkpoint(model, optimizer, scaler, checkpoint_path: Path, device):\n    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)\n    model.load_state_dict(checkpoint["model_state_dict"])\n    if optimizer is not None:\n        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])\n    if scaler is not None and checkpoint.get("scaler_state_dict") is not None:\n        scaler.load_state_dict(checkpoint["scaler_state_dict"])\n    return checkpoint["epoch"]\n\n\ndef parse_args():\n    parser = argparse.ArgumentParser(description="Train a larger diffusion model on CelebA 128x128")\n    parser.add_argument("--data-dir", type=str, default="./data")\n    parser.add_argument("--output-dir", type=str, default="./outputs_celeb")\n    parser.add_argument("--batch-size", type=int, default=64)\n    parser.add_argument("--epochs", type=int, default=20)\n    parser.add_argument("--lr", type=float, default=2e-4)\n    parser.add_argument("--num-workers", type=int, default=4)\n    parser.add_argument("--num-samples", type=int, default=16)\n    parser.add_argument("--sample-steps", type=int, default=50, help="DDIM denoising steps for preview images")\n    parser.add_argument("--num-res-blocks", type=int, default=2, help="ResNet blocks per down/up stage")\n    parser.add_argument("--num-mid-blocks", type=int, default=2, help="Bottleneck blocks at lowest resolution")\n    parser.add_argument("--checkpoint", type=str, default=None)\n    parser.add_argument("--sample-only", action="store_true")\n    parser.add_argument("--no-amp", action="store_true", help="Disable bf16 autocast even on CUDA")\n    return parser.parse_args()\n\n\ndef main():\n    args = parse_args()\n    device = get_device()\n    output_dir = Path(args.output_dir)\n    amp_enabled = use_bf16(device) and not args.no_amp\n\n    model = UnetLarge(\n        in_channels=IN_CHANNELS,\n        num_res_blocks=args.num_res_blocks,\n        num_mid_blocks=args.num_mid_blocks,\n    ).to(device)\n    scheduler = LinearNoiseScheduler(NUM_TIMESTEPS, BETA_START, BETA_END).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr)\n    scaler = None\n\n    param_count = sum(p.numel() for p in model.parameters())\n    print(f"Using device: {device}")\n    print(f"Model parameters: {param_count:,}")\n    print(f"bf16 autocast: {amp_enabled}")\n\n    start_epoch = 0\n    if args.checkpoint:\n        start_epoch = load_checkpoint(model, optimizer, scaler, Path(args.checkpoint), device)\n        print(f"Loaded checkpoint from epoch {start_epoch}")\n\n    if args.sample_only:\n        sample_images(\n            model,\n            scheduler,\n            device,\n            args.num_samples,\n            output_dir / "samples.png",\n            args.sample_steps,\n            amp_enabled,\n        )\n        print(f"Saved samples to {output_dir / \'samples.png\'}")\n        return\n\n    train_loader, test_loader = get_celeba_dataloaders(\n        args.data_dir,\n        args.batch_size,\n        args.num_workers,\n    )\n    print(\n        f"Loaded CelebA: {len(train_loader.dataset)} train, "\n        f"{len(test_loader.dataset)} valid images at {IMAGE_SIZE}x{IMAGE_SIZE}"\n    )\n\n    for epoch in range(start_epoch, args.epochs):\n        epoch_num = epoch + 1\n        train_loss = train_epoch(\n            model, train_loader, scheduler, optimizer, device, epoch_num, args.epochs, amp_enabled\n        )\n        test_loss = evaluate(\n            model, test_loader, scheduler, device, epoch_num, args.epochs, amp_enabled\n        )\n\n        tqdm.write(\n            f"Epoch {epoch_num}/{args.epochs} | "\n            f"train loss: {train_loss:.4f} | valid loss: {test_loss:.4f}"\n        )\n\n        save_checkpoint(model, optimizer, scaler, epoch_num, output_dir)\n        sample_images(\n            model,\n            scheduler,\n            device,\n            args.num_samples,\n            output_dir / f"samples_epoch_{epoch_num}.png",\n            args.sample_steps,\n            amp_enabled,\n        )\n\n    print(f"Training complete. Checkpoints and samples saved to {output_dir}")\n\n\nif __name__ == "__main__":\n    main()\n'
(REPO_DIR / 'train_celeb.py').write_text(TRAIN_CELEB_SOURCE)

command = [
    sys.executable, 'train_celeb.py',
    '--data-dir', str(LOCAL_DATA_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--batch-size', str(BATCH_SIZE),
    '--epochs', str(EPOCHS),
    '--lr', str(LEARNING_RATE),
    '--num-workers', str(NUM_WORKERS),
    '--num-samples', str(NUM_SAMPLES),
    '--sample-steps', str(SAMPLE_STEPS),
    '--num-res-blocks', str(NUM_RES_BLOCKS),
    '--num-mid-blocks', str(NUM_MID_BLOCKS),
]

if RESUME and CHECKPOINT_PATH.exists():
    command += ['--checkpoint', str(CHECKPOINT_PATH)]
if NO_AMP:
    command += ['--no-amp']

log_path = OUTPUT_DIR / 'train.log'
print('Installed train_celeb.py with CelebAImageDataset.')
print('Running:', ' '.join(command))
print(f'Log file: {log_path}')

environment = os.environ.copy()
environment['PYTHONUNBUFFERED'] = '1'
with log_path.open('a', encoding='utf-8') as log_file:
    log_file.write('\n\n=== New training run ===\n')
    log_file.write(' '.join(command) + '\n')
    log_file.flush()
    process = subprocess.Popen(
        command,
        cwd=REPO_DIR,
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
        log_file.flush()
    return_code = process.wait()

if return_code != 0:
    raise subprocess.CalledProcessError(return_code, command)
print('Training finished successfully.')

In [ ]:
from IPython.display import Image, display

assert CHECKPOINT_PATH.exists(), f'Missing checkpoint: {CHECKPOINT_PATH}'
sample_paths = sorted(OUTPUT_DIR.glob('samples_epoch_*.png'))
assert sample_paths, f'No sample images found in {OUTPUT_DIR}'

print(f'Checkpoint: {CHECKPOINT_PATH} ({CHECKPOINT_PATH.stat().st_size / 1024**2:.1f} MB)')
print(f'Log: {log_path}')
print(f'Samples: {len(sample_paths)}')
print('All artifacts are in Google Drive under:', DRIVE_ROOT)
display(Image(filename=str(sample_paths[-1])))

## Optional: generate samples later from the Drive checkpoint

Run this cell after a training run if you only want to regenerate `samples.png` without training.

In [ ]:
# Uncomment and run when you want sample-only generation.
# sample_command = [
#     sys.executable, 'train_celeb.py',
#     '--data-dir', str(LOCAL_DATA_DIR),
#     '--output-dir', str(OUTPUT_DIR),
#     '--checkpoint', str(CHECKPOINT_PATH),
#     '--sample-only',
#     '--num-samples', str(NUM_SAMPLES),
#     '--sample-steps', str(SAMPLE_STEPS),
# ]
# subprocess.run(sample_command, cwd=REPO_DIR, check=True)